In [1]:
import sys
print(sys.executable)

C:\Users\roaa1\anaconda3\envs\arsl2\python.exe


In [2]:
import sys
print(sys.executable)
import numpy; print(numpy.__version__)
import tensorflow as tf; print(tf.__version__)
import cv2; print(cv2.__version__)

C:\Users\roaa1\anaconda3\envs\arsl2\python.exe
1.23.5
2.12.0
4.8.0


# Combined ASL Translator + Word Mode + Tutor Mode

Modes:
- Press **1** = Letter translation mode
- Press **2** = Word recognition mode
- Press **3** = Tutor mode
- Press **N** = Next tutor guide (only in tutor mode)

Also includes Arabic + Spanish translation and distance checking.

In [3]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import tensorflow as tf
from tensorflow.python.keras.models import load_model
from tensorflow.keras.layers import BatchNormalization, Dropout, Dense, Flatten, Conv2D, MaxPooling2D
import numpy as np

# ==============================
# 1) LETTER MODEL
# ==============================
LETTER_MODEL_PATH_OPTIONS = [
    "asl_mediapipe_mlp_model.h5",
    "./asl_mediapipe_mlp_model.h5",
]
LETTER_MODEL_PATH = next((p for p in LETTER_MODEL_PATH_OPTIONS if os.path.exists(p)), None)
if LETTER_MODEL_PATH is None:
    raise FileNotFoundError("Letter model not found. Put asl_mediapipe_mlp_model.h5 inside Sign_to_Sentence Project folder.")

model = load_model(LETTER_MODEL_PATH, compile=False)
print("✅ Letter model loaded!")
print("Letter Input:", model.input_shape)
print("Letter Output:", model.output_shape)

# ==============================
# 2) WORD MODEL
# Required files:
#   - gesture_word_model.h5
#   - gesture_labels.npy
# You can keep them in: word model/ OR Sign_to_Sentence Project/ OR same folder as notebook
# ==============================
WORD_MODEL_OPTIONS = [
    r"word model\gesture_word_model.h5",
    r"word_model\word model\gesture_word_model.h5",
    r"Sign_to_Sentence Project\gesture_word_model.h5",
    r"gesture_word_model.h5"
]
WORD_LABEL_OPTIONS = [
    r"word model\gesture_labels.npy",
    r"word_model\word model\gesture_labels.npy",
    r"Sign_to_Sentence Project\gesture_labels.npy",
    r"gesture_labels.npy"
]

WORD_MODEL_PATH = next((p for p in WORD_MODEL_OPTIONS if os.path.exists(p)), None)
WORD_LABELS_PATH = next((p for p in WORD_LABEL_OPTIONS if os.path.exists(p)), None)

if WORD_MODEL_PATH is None or WORD_LABELS_PATH is None:
    raise FileNotFoundError(
        "Word model files not found. Extract 'word model.zip' and keep the folder next to this notebook, "
        "or copy gesture_word_model.h5 and gesture_labels.npy into Sign_to_Sentence Project."
    )

# Fix for: ValueError Unknown layer: BatchNormalization
custom_objects = {
    'BatchNormalization': BatchNormalization,
    'Dropout': Dropout,
    'Dense': Dense,
    'Flatten': Flatten,
    'Conv2D': Conv2D,
    'MaxPooling2D': MaxPooling2D,
}

word_model = load_model(WORD_MODEL_PATH, compile=False, custom_objects=custom_objects)
WORD_LABELS = np.load(WORD_LABELS_PATH, allow_pickle=True).tolist()
print("✅ Word model loaded!")
print("Word classes:", WORD_LABELS)


✅ Letter model loaded!
Letter Input: (None, 63)
Letter Output: (None, 28)
✅ Word model loaded!
Word classes: ['computer vision', 'good', 'hi', 'how are you', 'thank you']


In [4]:
import sys
!{sys.executable} -m pip install mediapipe==0.10.9 googletrans==4.0.0-rc1 arabic-reshaper==2.1.4 python-bidi pillow matplotlib pandas

In [5]:
import cv2
import numpy as np
import mediapipe as mp
import time
import warnings
import os
import io
warnings.filterwarnings('ignore')
from googletrans import Translator
from PIL import ImageFont, ImageDraw, Image
import arabic_reshaper
from bidi.algorithm import get_display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ── MediaPipe ──
mp_hands = mp.solutions.hands
mp_draw  = mp.solutions.drawing_utils
hands    = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ── Translator ──
translator = Translator()

# ── ASL Labels ──
ASL_LETTERS = ['A','B','C','D','E','F','G','H','I','J','K','L','M',
               'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
               'del','space']

# ── Settings ──
CONFIRM_SECONDS      = 1.5
CONFIDENCE_THRESHOLD = 0.60
STABILITY_FRAMES     = 3

# ── Arabic Font ──
ARABIC_FONT_PATH = r"C:\Users\roaa1\Amiri-Regular.ttf"

def put_arabic_text(frame, text, y_position, x_start_pos=None,
                    color=(255,200,100), fontsize=26):
    if not text or text == "---":
        return
    try:
        reshaped  = arabic_reshaper.reshape(text)
        bidi_text = get_display(reshaped)
        h, w      = frame.shape[:2]
        panel_w   = (w - 52) // 3
        dpi       = 100

        fig, ax = plt.subplots(figsize=(panel_w/dpi, 55/dpi), dpi=dpi)
        fig.patch.set_facecolor((15/255, 15/255, 15/255))
        ax.set_facecolor((15/255, 15/255, 15/255))
        ax.axis('off')

        if os.path.exists(ARABIC_FONT_PATH):
            prop = fm.FontProperties(fname=ARABIC_FONT_PATH, size=fontsize)
        else:
            prop = fm.FontProperties(size=fontsize)

        ax.text(0.98, 0.5, bidi_text,
                transform=ax.transAxes,
                fontproperties=prop,
                color=(color[0]/255, color[1]/255, color[2]/255),
                ha='right', va='center')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight',
                    facecolor=fig.get_facecolor())
        plt.close(fig)
        buf.seek(0)

        pil_img  = Image.open(buf).convert('RGB')
        buf_arr  = np.array(pil_img)
        buf_bgr  = cv2.cvtColor(buf_arr, cv2.COLOR_RGB2BGR)
        buf_bgr  = cv2.resize(buf_bgr, (panel_w, 55))

        x_start = x_start_pos if x_start_pos is not None else (26 + panel_w + 16)
        y_end   = y_position + 55
        if y_end < h and y_position >= 0:
            frame[y_position:y_end, x_start:x_start+panel_w] = buf_bgr

    except Exception as e:
        print(f"[Arabic error]: {e}")

def translate_sentence(english_text):
    """Translate English to Arabic AND Spanish."""
    if not english_text.strip():
        return "", ""

    clean = english_text.strip().lower()

    # Try Arabic separately
    arabic = ""
    try:
        result = translator.translate(clean, src='en', dest='ar')
        if result and result.text:
            arabic = result.text
            if arabic.lower() == clean:
                arabic = ""
    except Exception as e:
        print(f"[WARN] Arabic failed: {e}")
        arabic = ""

    # Try Spanish separately
    spanish = ""
    try:
        result = translator.translate(clean, src='en', dest='es')
        if result and result.text:
            spanish = result.text
            if spanish.lower() == clean:
                spanish = ""
    except Exception as e:
        print(f"[WARN] Spanish failed: {e}")
        spanish = ""

    print(f"[EN] {clean} | [AR] {arabic} | [ES] {spanish}")
    return arabic, spanish

print("✅ All set!")

✅ All set!


In [6]:
stability_buffer = []
word_stability_buffer = []

def extract_landmarks(hand_landmarks):
    landmarks = []
    for lm in hand_landmarks.landmark:
        landmarks.extend([lm.x, lm.y, lm.z])
    return np.array(landmarks).reshape(1, -1)

def is_open_palm(hand_landmarks):
    tips = [4, 8, 12, 16, 20]
    mcps = [2, 5,  9, 13, 17]
    lm   = hand_landmarks.landmark
    up   = sum(1 for t, m in zip(tips, mcps) if lm[t].y < lm[m].y)
    return up >= 4

def get_stable_letter(detected):
    global stability_buffer
    stability_buffer.append(detected)
    if len(stability_buffer) > STABILITY_FRAMES:
        stability_buffer.pop(0)
    if len(stability_buffer) == STABILITY_FRAMES:
        if len(set(stability_buffer)) == 1:
            return stability_buffer[0]
    return None

def get_hand_distance(hlm):
    xs = [lm.x for lm in hlm.landmark]
    ys = [lm.y for lm in hlm.landmark]
    size = max(max(xs)-min(xs), max(ys)-min(ys))
    if size > 0.55:  return 'too_close'
    if size < 0.18:  return 'too_far'
    return 'ok'


def get_stable_word(detected):
    global word_stability_buffer
    word_stability_buffer.append(detected)
    if len(word_stability_buffer) > STABILITY_FRAMES:
        word_stability_buffer.pop(0)
    if len(word_stability_buffer) == STABILITY_FRAMES:
        if len(set(str(x) for x in word_stability_buffer)) == 1:
            return word_stability_buffer[0]
    return None

def predict_word_from_landmarks(hand_landmarks):
    landmarks = extract_landmarks(hand_landmarks).astype(np.float32)
    preds = word_model(landmarks, training=False).numpy()[0]
    top_idx = int(np.argmax(preds))
    confidence = float(preds[top_idx])
    predicted_word = WORD_LABELS[top_idx]
    return predicted_word, confidence

def draw_rounded_rect(frame, x1, y1, x2, y2, color, thickness=2, r=10):
    cv2.line(frame, (x1+r, y1), (x2-r, y1), color, thickness)
    cv2.line(frame, (x1+r, y2), (x2-r, y2), color, thickness)
    cv2.line(frame, (x1, y1+r), (x1, y2-r), color, thickness)
    cv2.line(frame, (x2, y1+r), (x2, y2-r), color, thickness)
    cv2.ellipse(frame, (x1+r, y1+r), (r,r), 180, 0, 90, color, thickness)
    cv2.ellipse(frame, (x2-r, y1+r), (r,r), 270, 0, 90, color, thickness)
    cv2.ellipse(frame, (x1+r, y2-r), (r,r),  90, 0, 90, color, thickness)
    cv2.ellipse(frame, (x2-r, y2-r), (r,r),   0, 0, 90, color, thickness)

def draw_card(frame, x1, y1, x2, y2, alpha=0.82):
    overlay = frame.copy()
    cv2.rectangle(overlay, (x1,y1), (x2,y2), (18,18,24), -1)
    cv2.addWeighted(overlay, alpha, frame, 1-alpha, 0, frame)
    draw_rounded_rect(frame, x1, y1, x2, y2, (55,55,75), thickness=1)

def draw_conf_bar(frame, x, y, w, val, color=(80,200,120)):
    cv2.rectangle(frame, (x,y), (x+w, y+5), (40,40,55), -1)
    filled = int(w * min(val, 1.0))
    if filled > 0:
        cv2.rectangle(frame, (x,y), (x+filled, y+5), color, -1)

def draw_ui(frame, state):
    h, w = frame.shape[:2]

    # ── TOP LEFT — Title card ──
    draw_card(frame, 12, 12, 420, 100)
    mode_name = state.get('mode', 'letter').upper()
    cv2.putText(frame, f"ASL -> AR/ES | {mode_name}",
                (26, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)
    cv2.putText(frame, "AI385 — Computer Vision",
                (26, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (120,120,160), 1)
    dot_color = (80,220,120) if (state['current_letter'] or state.get('current_gesture') or state.get('mode') == 'tutor') else (80,80,100)
    cv2.circle(frame, (388, 45), 8, dot_color, -1)
    cv2.putText(frame, "LIVE" if (state['current_letter'] or state.get('current_gesture') or state.get('mode') == 'tutor') else "IDLE",
                (402, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.38, dot_color, 1)

    # ── TOP RIGHT — Detection card ──
    draw_card(frame, w-310, 12, w-12, 215)

    if state['current_letter'] and state['confidence'] > 0:
        conf = state['confidence']
        pct  = int(conf * 100)
        col  = (80,220,120) if pct>=80 else (80,180,255) if pct>=65 else (200,160,60)

        cv2.putText(frame, state['current_letter'],
                    (w-285, 90), cv2.FONT_HERSHEY_SIMPLEX, 2.5, col, 4)
        cv2.putText(frame, f"Detected: {pct}%",
                    (w-285, 118), cv2.FONT_HERSHEY_SIMPLEX, 0.52, col, 1)
        draw_conf_bar(frame, w-285, 130, 255, conf, col)

        if state['stable_letter']:
            cv2.putText(frame, f"Stable: {state['stable_letter']}",
                        (w-285, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (80,220,120), 1)
            draw_conf_bar(frame, w-285, 170, 255,
                          state['hold_progress'], (80,220,120))
            cv2.putText(frame, f"Hold: {int(state['hold_progress']*100)}%",
                        (w-285, 195), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100,100,130), 1)
        else:
            cv2.putText(frame, "Keep hand still...",
                        (w-285, 162), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100,100,130), 1)
    elif state.get('mode') == 'tutor':
        cv2.putText(frame, "Tutor Mode",
                    (w-285, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (120,220,255), 2)
        cv2.putText(frame, f"Target: {state.get('tutor_target','---')}",
                    (w-285, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
        cv2.putText(frame, "Press N for next guide",
                    (w-285, 142), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100,100,130), 1)
    else:
        cv2.putText(frame, "No sign detected",
                    (w-285, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (80,80,110), 1)
        cv2.putText(frame, "Show hand to camera",
                    (w-285, 118), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (60,60,90), 1)

    # ── BOTTOM PANEL ──
    draw_card(frame, 12, h-240, w-12, h-12)

    current_word = ''.join(state['current_word'])
    full_en      = ' '.join(state['words'])
    if current_word:
        full_en = (full_en + ' ' + current_word).strip() if full_en else current_word

    # Word being signed
    cv2.putText(frame, "WORD:",
                (26, h-212), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100,100,140), 1)
    cv2.putText(frame, current_word if current_word else "---",
                (26, h-178), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255,255,255), 2)

    cv2.line(frame, (26, h-158), (w-26, h-158), (45,45,65), 1)

    # Calculate third widths
    third = (w - 52) // 3

    # ── Left third — English ──
    cv2.putText(frame, "ENGLISH:",
                (26, h-135), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100,100,140), 1)
    words_list = full_en.strip().split()
    line1 = ' '.join(words_list[:4])
    line2 = ' '.join(words_list[4:])
    cv2.putText(frame, line1 if line1 else "---",
                (26, h-108), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (100,200,255), 2)
    if line2:
        cv2.putText(frame, line2,
                    (26, h-78), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (100,200,255), 2)

    # Divider 1
    cv2.line(frame, (26+third, h-155), (26+third, h-25), (45,45,65), 1)

    # ── Middle third — Arabic ──
    mid_x = 26 + third + 16
    cv2.putText(frame, "ARABIC:",
                (mid_x, h-135), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100,100,140), 1)
    if state['arabic']:
        put_arabic_text(frame, state['arabic'],
                        h-128, x_start_pos=mid_x,
                        color=(255,200,100), fontsize=26)
    else:
        cv2.putText(frame, "---",
                    (mid_x, h-100), cv2.FONT_HERSHEY_SIMPLEX,
                    0.65, (60,60,80), 1)

    # Divider 2
    cv2.line(frame, (26+third*2, h-155), (26+third*2, h-25), (45,45,65), 1)

    # ── Right third — Spanish ──
    right_x = 26 + third*2 + 16
    cv2.putText(frame, "SPANISH:",
                (right_x, h-135), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (100,100,140), 1)
    cv2.putText(frame, state.get('spanish', '') or "---",
                (right_x, h-100), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (120,220,160), 2)

    # Status + controls
    total = sum(len(wd) for wd in state['words']) + len(state['current_word'])
    cv2.putText(frame,
                f"{total} letters  •  {len(state.get('words',[]))} words  •  {state.get('status','')}",
                (26, h-22), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (70,70,95), 1)
    cv2.putText(frame,
                "1=letter | 2=word | 3=tutor | N=next | ENTER=translate | C=clear | Q=quit",
                (w-460, h-22), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (60,60,85), 1)

    # ── Distance / visibility banners ──
    dist = state.get('distance', 'ok')
    px   = w - 310
    if dist == 'too_close':
        dmsg, dbg = '<<  Put a little space!  >>', (0, 0, 200)
    elif dist == 'too_far':
        dmsg, dbg = '>>  Come closer!  <<', (0, 140, 0)
    else:
        dmsg, dbg = None, None
    if dmsg:
        cv2.rectangle(frame, (0, h//2-45), (px, h//2+25), dbg, -1)
        dfs = 1.25
        tw  = cv2.getTextSize(dmsg, cv2.FONT_HERSHEY_DUPLEX, dfs, 3)[0][0]
        cv2.putText(frame, dmsg, ((px-tw)//2, h//2),
                    cv2.FONT_HERSHEY_DUPLEX, dfs, (255,255,255), 3)

print("✅ UI ready!")

# ── Tutor Mode Helpers ──
import zipfile
import random

TUTOR_FOLDER = "asl_tutor_guides"
TUTOR_ZIP = "asl_tutor_guides.zip"

# If the tutor folder is missing but the zip exists, extract it automatically.
if not os.path.exists(TUTOR_FOLDER) and os.path.exists(TUTOR_ZIP):
    with zipfile.ZipFile(TUTOR_ZIP, 'r') as zip_ref:
        zip_ref.extractall('.')

def load_guide_image(letter):
    path = os.path.join(TUTOR_FOLDER, f"{letter}.png")
    guide = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    return guide


def overlay_guide(frame, guide):
    """Show transparent ASL guide image on the camera frame."""
    if guide is None:
        return frame

    guide = cv2.resize(guide, (300, 380))
    h, w = guide.shape[:2]

    # Put guide near the middle-left, avoiding the right UI card.
    x = 450
    y = 110

    if y + h > frame.shape[0] or x + w > frame.shape[1]:
        return frame

    if len(guide.shape) == 3 and guide.shape[2] == 4:
        alpha = guide[:, :, 3] / 255.0
        for c in range(3):
            frame[y:y+h, x:x+w, c] = (
                alpha * guide[:, :, c] +
                (1 - alpha) * frame[y:y+h, x:x+w, c]
            )
    else:
        overlay = frame.copy()
        overlay[y:y+h, x:x+w] = guide[:, :, :3]
        cv2.addWeighted(overlay, 0.35, frame, 0.65, 0, frame)

    return frame


def next_tutor_letter(state):
    letters = list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')
    current = state.get('tutor_target')
    if current in letters:
        idx = (letters.index(current) + 1) % len(letters)
    else:
        idx = 0
    state['tutor_target'] = letters[idx]
    state['tutor_guide'] = load_guide_image(state['tutor_target'])
    state['correct_frames'] = 0
    state['status'] = f"Tutor target: {state['tutor_target']}"

print("✅ Tutor helpers ready!")


✅ UI ready!
✅ Tutor helpers ready!


In [8]:
stability_buffer = []
word_stability_buffer = []

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

state = {
    'mode'           : 'letter',   # 1 = letter, 2 = word, 3 = tutor
    'current_word'   : [],
    'words'          : [],
    'current_letter' : None,
    'stable_letter'  : None,
    'current_gesture': None,
    'stable_gesture' : None,
    'confidence'     : 0.0,
    'hold_start'     : None,
    'hold_progress'  : 0.0,
    'last_confirmed' : None,
    '_holding_letter': None,
    'arabic'         : '',
    'spanish'        : '',
    'status'         : 'Ready — press 1 letters, 2 words, 3 tutor!',
    'distance'       : 'ok',
    'tutor_target'   : 'A',
    'tutor_guide'    : load_guide_image('A'),
    'correct_frames' : 0,
}

print("✅ Starting — window will open on your desktop!")
print("Press 1 = LETTER mode")
print("Press 2 = WORD mode")
print("Press 3 = TUTOR mode")
print("Press N = next tutor guide")
print("ENTER = translate to Arabic & Spanish | C = clear | Q = quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame  = cv2.flip(frame, 1)
    rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    detected_letter = None
    detected_word   = None
    confidence      = 0.0

    if result.multi_hand_landmarks:
        hand_lm = result.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(frame, hand_lm, mp_hands.HAND_CONNECTIONS)
        state['distance'] = get_hand_distance(hand_lm)

        # =========================================================
        # MODE 1: LETTER MODEL
        # =========================================================
        if state['mode'] == 'letter':
            landmarks  = extract_landmarks(hand_lm)
            preds      = model(landmarks, training=False).numpy()[0]
            top_idx    = int(np.argmax(preds))
            confidence = float(preds[top_idx])

            if confidence >= CONFIDENCE_THRESHOLD:
                detected_letter          = ASL_LETTERS[top_idx]
                state['current_letter']  = detected_letter
                state['current_gesture'] = None
                state['confidence']      = confidence

                col = (80,220,120) if confidence >= 0.80 else (80,180,255)
                cv2.putText(frame, detected_letter,
                            (30, 120), cv2.FONT_HERSHEY_SIMPLEX,
                            4.0, col, 7)
                cv2.putText(frame, f"{int(confidence*100)}%",
                            (30, 160), cv2.FONT_HERSHEY_SIMPLEX,
                            1.0, (150,150,150), 2)
            else:
                state['current_letter'] = None
                state['stable_letter']  = None
                stability_buffer.clear()

        # =========================================================
        # MODE 2: WORD MODEL
        # =========================================================
        elif state['mode'] == 'word':
            detected_word, confidence = predict_word_from_landmarks(hand_lm)
            state['current_gesture'] = detected_word
            state['current_letter']  = None
            state['confidence']      = confidence

            col = (120,220,160) if confidence >= 0.85 else (80,180,255)
            cv2.putText(frame, detected_word.upper(),
                        (30, 120), cv2.FONT_HERSHEY_SIMPLEX,
                        1.8, col, 4)
            cv2.putText(frame, f"{int(confidence*100)}%",
                        (30, 160), cv2.FONT_HERSHEY_SIMPLEX,
                        1.0, (150,150,150), 2)

        # =========================================================
        # MODE 3: TUTOR MODE
        # Shows target image and checks if your predicted letter matches it.
        # =========================================================
        elif state['mode'] == 'tutor':
            landmarks  = extract_landmarks(hand_lm)
            preds      = model(landmarks, training=False).numpy()[0]
            top_idx    = int(np.argmax(preds))
            confidence = float(preds[top_idx])

            if confidence >= CONFIDENCE_THRESHOLD:
                detected_letter          = ASL_LETTERS[top_idx]
                state['current_letter']  = detected_letter
                state['current_gesture'] = None
                state['confidence']      = confidence

                target = state.get('tutor_target', 'A')
                is_correct = (detected_letter == target)
                if is_correct:
                    state['correct_frames'] += 1
                    state['status'] = f"Correct! You signed {target}"
                    col = (80,220,120)
                else:
                    state['correct_frames'] = 0
                    state['status'] = f"Try {target} — detected {detected_letter}"
                    col = (0,180,255)

                cv2.putText(frame, f"Target: {target}",
                            (30, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255,255,255), 3)
                cv2.putText(frame, f"Detected: {detected_letter} ({int(confidence*100)}%)",
                            (30, 155), cv2.FONT_HERSHEY_SIMPLEX, 0.9, col, 2)

                if state['correct_frames'] >= 15:
                    cv2.putText(frame, "GOOD JOB! Press N for next",
                                (30, 205), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (80,220,120), 2)
            else:
                state['current_letter'] = None
                state['confidence'] = 0.0
                state['correct_frames'] = 0
                state['status'] = f"Show letter {state.get('tutor_target','A')}"

        # Bounding box for all modes
        h_f, w_f = frame.shape[:2]
        xs = [lm.x * w_f for lm in hand_lm.landmark]
        ys = [lm.y * h_f for lm in hand_lm.landmark]
        x1 = max(int(min(xs)) - 25, 0)
        x2 = min(int(max(xs)) + 25, w_f)
        y1 = max(int(min(ys)) - 25, 0)
        y2 = min(int(max(ys)) + 25, h_f)
        draw_rounded_rect(frame, x1, y1, x2, y2, (80,220,120), thickness=2)

    else:
        state['current_letter']  = None
        state['stable_letter']   = None
        state['current_gesture'] = None
        state['stable_gesture']  = None
        state['hold_start']      = None
        state['hold_progress']   = 0.0
        state['status']          = 'No hand detected'
        state['distance']        = 'ok'
        stability_buffer.clear()
        word_stability_buffer.clear()

        h_f, w_f = frame.shape[:2]
        px  = w_f - 310
        msg = '>>  Show your hand!  <<'
        cv2.rectangle(frame, (0, h_f//2-45), (px, h_f//2+25), (0,100,180), -1)
        dfs = 1.3
        tw  = cv2.getTextSize(msg, cv2.FONT_HERSHEY_DUPLEX, dfs, 3)[0][0]
        cv2.putText(frame, msg, ((px-tw)//2, h_f//2),
                    cv2.FONT_HERSHEY_DUPLEX, dfs, (255,255,255), 3)

    # =============================================================
    # CONFIRMATION LOGIC — LETTER MODE
    # =============================================================
    if state['mode'] == 'letter' and detected_letter:
        stable = get_stable_letter(detected_letter)
        state['stable_letter'] = stable

        if stable:
            state['status'] = f'Hold still: {stable}'

            if state['_holding_letter'] != stable or state['hold_start'] is None:
                state['hold_start']      = time.time()
                state['_holding_letter'] = stable
                state['last_confirmed']  = None

            held = time.time() - state['hold_start']
            state['hold_progress'] = min(held / CONFIRM_SECONDS, 1.0)

            if held >= CONFIRM_SECONDS and state['last_confirmed'] != stable:

                if stable == 'space':
                    if state['current_word']:
                        word = ''.join(state['current_word'])
                        state['words'].append(word)
                        state['current_word'] = []
                        state['status']       = f'Word: {word}'
                        print(f"[SPACE] word: {word}")
                        full_en = ' '.join(state['words'])
                        state['arabic'], state['spanish'] = translate_sentence(full_en)

                elif stable == 'del':
                    if state['current_word']:
                        removed = state['current_word'].pop()
                        state['status'] = f'Deleted: {removed}'
                        print(f"[DEL] {removed}")
                    elif state['words']:
                        last = state['words'].pop()
                        state['current_word'] = list(last)
                        state['status'] = 'Restored last word'

                else:
                    state['current_word'].append(stable)
                    state['status'] = f'✓ Added letter: {stable}'
                    print(f"[+] {stable} | word: {''.join(state['current_word'])}")

                state['last_confirmed']  = stable
                state['_holding_letter'] = None
                state['hold_start']      = None
                state['hold_progress']   = 0.0
                stability_buffer.clear()
        else:
            state['hold_start']    = None
            state['hold_progress'] = 0.0
            state['status']        = f'Keep still... ({detected_letter})'

    # =============================================================
    # CONFIRMATION LOGIC — WORD MODE
    # =============================================================
    elif state['mode'] == 'word' and detected_word and confidence >= 0.85:
        stable_word = get_stable_word(detected_word)
        state['stable_gesture'] = stable_word

        if stable_word:
            state['status'] = f'Hold word: {stable_word}'

            if state['_holding_letter'] != stable_word or state['hold_start'] is None:
                state['hold_start']      = time.time()
                state['_holding_letter'] = stable_word
                state['last_confirmed']  = None

            held = time.time() - state['hold_start']
            state['hold_progress'] = min(held / 0.6, 1.0)

            if held >= 0.6 and state['last_confirmed'] != stable_word:
                state['words'].append(stable_word)
                state['current_word'] = []
                full_en = ' '.join(state['words'])
                state['arabic'], state['spanish'] = translate_sentence(full_en)
                state['status'] = f'✓ Added word: {stable_word}'
                print(f"[WORD] {stable_word}")

                state['last_confirmed']  = stable_word
                state['_holding_letter'] = None
                state['hold_start']      = None
                state['hold_progress']   = 0.0
                word_stability_buffer.clear()
        else:
            state['hold_start']    = None
            state['hold_progress'] = 0.0
            state['status']        = f'Keep still... ({detected_word})'

    elif state['mode'] != 'tutor':
        state['hold_start']    = None
        state['hold_progress'] = 0.0

    # Show tutor guide image after camera processing, before UI/window display.
    if state['mode'] == 'tutor':
        frame = overlay_guide(frame, state.get('tutor_guide'))

    draw_ui(frame, state)
    cv2.imshow("ASL Arabic Spanish Translator — Letter / Word / Tutor Modes", frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

    elif key == ord('1'):
        state['mode'] = 'letter'
        state['status'] = 'Switched to LETTER mode'
        state['current_letter'] = None
        state['current_gesture'] = None
        state['hold_start'] = None
        state['hold_progress'] = 0.0
        stability_buffer.clear()
        word_stability_buffer.clear()
        print("[MODE] Letter mode")

    elif key == ord('2'):
        state['mode'] = 'word'
        state['status'] = 'Switched to WORD mode'
        state['current_letter'] = None
        state['current_gesture'] = None
        state['hold_start'] = None
        state['hold_progress'] = 0.0
        stability_buffer.clear()
        word_stability_buffer.clear()
        print("[MODE] Word mode")

    elif key == ord('3'):
        state['mode'] = 'tutor'
        state['status'] = f"Switched to TUTOR mode — target {state.get('tutor_target','A')}"
        state['current_letter'] = None
        state['current_gesture'] = None
        state['hold_start'] = None
        state['hold_progress'] = 0.0
        state['correct_frames'] = 0
        stability_buffer.clear()
        word_stability_buffer.clear()
        print("[MODE] Tutor mode")

    elif key == ord('n'):
        if state['mode'] == 'tutor':
            next_tutor_letter(state)
            print(f"[TUTOR] Next target: {state['tutor_target']}")

    elif key == 13:  # ENTER
        full_en = ' '.join(state['words'])
        if state['current_word']:
            cw      = ''.join(state['current_word'])
            full_en = (full_en + ' ' + cw).strip() if full_en else cw
        print(f"[ENTER] translating: '{full_en}'")
        if full_en.strip():
            state['arabic'], state['spanish'] = translate_sentence(full_en)
            state['status'] = '✓ Translated!'

    elif key == 8:  # BACKSPACE
        if state['current_word']:
            removed = state['current_word'].pop()
            state['last_confirmed']  = None
            state['_holding_letter'] = None
            stability_buffer.clear()
            state['status'] = f'Deleted: {removed}'
        elif state['words']:
            last = state['words'].pop()
            state['status'] = f'Deleted word: {last}'
            state['arabic'], state['spanish'] = translate_sentence(' '.join(state['words'])) if state['words'] else ('', '')

    elif key == ord('c'):
        state.update({
            'current_word':[], 'words':[], 'arabic':'', 'spanish':'',
            'current_letter':None, 'stable_letter':None,
            'current_gesture':None, 'stable_gesture':None,
            'last_confirmed':None, '_holding_letter':None,
            'hold_start':None, 'hold_progress':0.0,
            'distance':'ok',
            'correct_frames':0,
            'status':'Cleared!'
        })
        stability_buffer.clear()
        word_stability_buffer.clear()
        print("[CLEAR]")

cap.release()
cv2.destroyAllWindows()
print("✅ Done!")


✅ Starting — window will open on your desktop!
Press 1 = LETTER mode
Press 2 = WORD mode
Press 3 = TUTOR mode
Press N = next tutor guide
ENTER = translate to Arabic & Spanish | C = clear | Q = quit
[MODE] Word mode
[WARN] Spanish failed: 'NoneType' object is not iterable
[EN] good | [AR] جيد | [ES] 
[WORD] good
[WARN] Spanish failed: 'NoneType' object is not iterable
[EN] good good | [AR] جيد جيد | [ES] 
[WORD] good
[WARN] Spanish failed: 'NoneType' object is not iterable
[EN] good good good | [AR] جيد جيد جيد | [ES] 
[WORD] good
[MODE] Tutor mode
[TUTOR] Next target: B
[TUTOR] Next target: C
[TUTOR] Next target: D
✅ Done!
